# Luna 生图工作台（Colab GPU + ComfyUI）

在浏览器里直接用 ComfyUI 的完整界面，GPU 是 Colab 免费的 T4。

**用法：菜单「代码执行程序」→「全部运行」，然后往下滚到最后一个单元格，界面就嵌在那里。**

顺序：
1. 第 1 格装环境（首次约 6~7 分钟，同一个会话内只需一次）
2. 第 2 格启动 ComfyUI 并把界面嵌进本页
3. 第 3 格（可选）传入定妆图，自动裁好 IP-Adapter 要的两张图

> 会话最长约 12 小时后会被回收，重开一个会话重跑前两格即可。


In [ ]:
import os, re, subprocess, sys, time

COMFY = "/content/ComfyUI"
NODES = f"{COMFY}/custom_nodes"
MODELS = f"{COMFY}/models"
SKIP = {"torch", "torchvision", "torchaudio", "xformers"}

def log(m): print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)
def run(args, cwd=None): subprocess.run(args, cwd=cwd, check=True)
def pkg(line): return re.split(r"[<>=!~\[;\s]", line.strip(), 1)[0].lower()

def clone(url, dest):
    if os.path.isdir(dest): log(f"已存在 {dest}"); return
    log(f"clone {url}"); run(["git", "clone", "--depth", "1", url, dest])

def pip_reqs(path, cwd):
    if not os.path.isfile(path): return
    want = [l.strip() for l in open(path) if l.strip() and not l.strip().startswith("#")
            and pkg(l) not in SKIP]
    if want:
        log(f"pip install {len(want)} 个依赖")
        run([sys.executable, "-m", "pip", "install", "-q", *want], cwd=cwd)

def place(repo, filename, dest_dir, dest_name=None):
    from huggingface_hub import hf_hub_download
    os.makedirs(dest_dir, exist_ok=True)
    dest = os.path.join(dest_dir, dest_name or os.path.basename(filename))
    if os.path.exists(dest): log(f"已存在 {os.path.basename(dest)}"); return dest
    src = hf_hub_download(repo_id=repo, filename=filename)
    os.symlink(src, dest); log(f"就位 {os.path.basename(dest)}"); return dest

log("=== 1/3 ComfyUI 本体 ===")
clone("https://github.com/comfyanonymous/ComfyUI.git", COMFY)
pip_reqs(f"{COMFY}/requirements.txt", COMFY)

log("=== 2/3 节点包 ===")
for url in ("https://github.com/cubiq/ComfyUI_IPAdapter_plus.git",
            "https://github.com/Fannovel16/comfyui_controlnet_aux.git"):
    dest = os.path.join(NODES, os.path.basename(url)[:-4])
    clone(url, dest); pip_reqs(f"{dest}/requirements.txt", dest)

log("=== 3/3 模型权重 ===")
place("OnomaAIResearch/Illustrious-XL-v2.0", "Illustrious-XL-v2.0.safetensors",
      f"{MODELS}/checkpoints")
for f in ("ip-adapter-plus_sdxl_vit-h.safetensors", "ip-adapter-plus-face_sdxl_vit-h.safetensors"):
    place("h94/IP-Adapter", f"sdxl_models/{f}", f"{MODELS}/ipadapter")
# 注意：models/image_encoder 才是 ViT-H-14（sdxl_models/image_encoder 是 ViT-bigG，别拿错）
place("h94/IP-Adapter", "models/image_encoder/model.safetensors", f"{MODELS}/clip_vision",
      "CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors")
log("SETUP_DONE —— 下一步运行「启动 ComfyUI」那个单元格")

In [ ]:
import subprocess, sys, time, urllib.request

if subprocess.run(["pgrep", "-f", "main.py --listen"], capture_output=True).returncode != 0:
    log = open("/content/comfy.log", "wb")
    subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188"],
                     cwd="/content/ComfyUI", stdout=log, stderr=subprocess.STDOUT,
                     stdin=subprocess.DEVNULL, start_new_session=True)
    print("正在启动 ComfyUI…")

for _ in range(90):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        print("ComfyUI 就绪，界面正在下方加载 ↓")
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("ComfyUI 启动失败，看 /content/comfy.log")

# 把 VM 里的 8188 端口嵌进本页 —— 这是 Colab 官方给本地服务做界面的方式
from google.colab import output
output.serve_kernel_port_as_iframe(8188, path="/", height="900")

In [ ]:
# 可选：把 Luna 的定妆图传进来，裁成 IP-Adapter 需要的头部/半身两张
import os
from google.colab import files
from PIL import Image

os.makedirs("/content/ref", exist_ok=True)
print("选择你的定妆图（全身立绘，白底最好）")
uploaded = files.upload()
src = Image.open(next(iter(uploaded.values())))
src = src.convert("RGB")
w, h = src.size
print("原图尺寸", src.size)

# 按比例裁：头部给 ip-adapter-plus-face，半身给服装适配器
src.crop((int(w*0.33), int(h*0.04), int(w*0.66), int(h*0.29))).save("/content/ref/face.png")
src.crop((int(w*0.14), int(h*0.04), int(w*0.87), int(h*0.67))).save("/content/ref/bust.png")
print("已生成 /content/ref/face.png 与 /content/ref/bust.png")
print("在 ComfyUI 里用「LoadImage」节点选它们即可")